# Atividade: Estatistica de Vendas com Python e Pandas

Este notebook recria automaticamente o `vendas.xlsx`, calcula os KPIs e gera os graficos solicitados.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ARQUIVO_EXCEL = Path("vendas.xlsx")
ARQUIVO_BARRAS = Path("grafico_barras.png")
ARQUIVO_BOXPLOT = Path("boxplot_vendas.png")

valores_venda = [
    1200, 1350, 1500, 1600, 1700,
    1800, 1900, 2000, 2100, 2200,
    2300, 2400, 2500, 2600, 9000,
]

vendedores = [f"Vendedor {i:02d}" for i in range(1, 16)]
df_base = pd.DataFrame({"Vendedor": vendedores, "Valor_Venda": valores_venda})
df_base.to_excel(ARQUIVO_EXCEL, index=False)
print(f"Arquivo Excel criado/recriado: {ARQUIVO_EXCEL.name}")
df_base.head()

In [ ]:
df = pd.read_excel(ARQUIVO_EXCEL)
df.columns = df.columns.str.strip()

colunas_obrigatorias = {"Vendedor", "Valor_Venda"}
faltando = colunas_obrigatorias - set(df.columns)
if faltando:
    raise ValueError(f"Colunas obrigatorias ausentes no Excel: {sorted(faltando)}")

df["Valor_Venda"] = (
    df["Valor_Venda"].astype(str)
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df["Valor_Venda"] = pd.to_numeric(df["Valor_Venda"], errors="coerce")

if df["Valor_Venda"].isna().any():
    raise ValueError("Existem valores invalidos na coluna 'Valor_Venda' apos conversao.")

df.head()

In [ ]:
def formatar_brl(valor: float) -> str:
    texto = f"{valor:,.2f}"
    return f"R$ {texto.replace(',', 'X').replace('.', ',').replace('X', '.')}"

media = df["Valor_Venda"].mean()
mediana = df["Valor_Venda"].median()
moda = df["Valor_Venda"].mode().tolist()
quartis = df["Valor_Venda"].quantile([0.25, 0.75])

print("KPIs de Vendas")
print(f"- Media: {formatar_brl(media)}")
print(f"- Mediana: {formatar_brl(mediana)}")
if moda:
    print("- Moda:", ", ".join(formatar_brl(v) for v in moda))
else:
    print("- Moda: Sem moda")
print(f"- Quartil 25% (Q1): {formatar_brl(quartis.loc[0.25])}")
print(f"- Quartil 75% (Q3): {formatar_brl(quartis.loc[0.75])}")

if media > mediana:
    print("\nObservacao: o outlier de R$ 9.000,00 puxa a media para cima.")

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))
plt.bar(df["Vendedor"], df["Valor_Venda"], color="#1f77b4")
plt.title("Desempenho Individual por Vendedor")
plt.xlabel("Vendedor")
plt.ylabel("Valor da Venda (R$)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(ARQUIVO_BARRAS, dpi=300)
plt.show()
print(f"Grafico salvo: {ARQUIVO_BARRAS.name}")

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df["Valor_Venda"], color="#ff9f43")
plt.title("Boxplot de Valor de Venda (detalhe de outliers)")
plt.xlabel("Valor da Venda (R$)")
plt.tight_layout()
plt.savefig(ARQUIVO_BOXPLOT, dpi=300)
plt.show()
print(f"Grafico salvo: {ARQUIVO_BOXPLOT.name}")